# Medição da calibração ampliada — 61 pares

Mede no BERTimbau os 61 pares de calibração aprovados em 14/09/2026 (condição
`calibracao_v2` de `experimentos/teste_explicito.py`) e refaz as duas análises que
dependem do grupo de referência: a reta da frequência do passo 5.4 e a análise de
direção do passo 5.5.

## O que este notebook faz, e o que não faz

**Faz.** Remede pares já medidos para conferir que o ambiente reproduz os números
guardados; mede os 61 pares novos, e só eles, porque `teste_explicito.py` pula as
condições já presentes em `explicito_bruto.json`; aplica a exclusão de
`controle_frequencia-05`; regera as tabelas; e exporta os arquivos para o Drive e
para download.

**Não faz.** Não altera o repositório no GitHub, não regera `pares_minimos.json`, não
atualiza `meta_pares_minimos.py` — cujos valores estão fixados no código e dependem
do desvio-padrão que a seção 6 produz — e não corrige os documentos que citam os
números antigos. Tudo isso é feito na máquina local, depois
(`docs/pendencias.md` 2.8).

## Por que a conferência da seção 4 vem antes da medição

As medições guardadas foram feitas em processador (`torch 2.13.0+cpu`,
`requirements-lock.txt`), e o Colab mede em GPU, com outra versão do `torch`. Se os
dois ambientes derem números diferentes, a reta seria ajustada sobre medições
misturadas, e a diferença apareceria como resultado. A seção 4 remede os cinco
pares do controle neutro e compara; a seção 5 recusa-se a rodar se a comparação
falhar.

## Antes de executar

- **Ambiente.** *Ambiente de execução* → *Alterar o tipo de ambiente* → **T4 GPU**.
  A GPU é recomendada, e não obrigatória: sem ela a medição leva alguns minutos a
  mais.
- **Nenhum token é necessário.** O modelo é público.
- **Execute as seções em ordem**, de 1 a 8. As instruções se referem às seções pelo
  título, e não pela posição da célula.

## 1. Instalação

In [ ]:
# Versoes de requirements-lock.txt, com que as medicoes guardadas foram feitas.
# O torch fica o do Colab, que traz suporte a GPU; a secao 4 confere se isso
# altera os numeros.
!pip install -q "transformers==5.16.1" "tokenizers==0.23.1" "huggingface-hub==1.26.1" "wordfreq==3.1.1"
print("instalado; siga para a secao 2")

## 2. Verificação do ambiente

Se alguma versão divergir da esperada, reinicie a sessão em *Ambiente de execução* →
*Reiniciar sessão* e execute de novo esta seção — a instalação permanece.

In [ ]:
from importlib.metadata import version
import torch

ESPERADO = {"transformers": "5.16.1", "tokenizers": "0.23.1",
            "huggingface-hub": "1.26.1", "wordfreq": "3.1.1"}
VERSOES = {pacote: version(pacote) for pacote in ESPERADO}
VERSOES["torch"] = torch.__version__

ok = True
for pacote, v in VERSOES.items():
    alvo = ESPERADO.get(pacote)
    confere = alvo is None or v == alvo
    ok &= confere
    print(f"{'OK   ' if confere else 'FALHA'}  {pacote:16s} {v}" + ("" if confere else f"  (esperado {alvo})"))

GPU_NOME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(f"\nGPU: {GPU_NOME or 'nenhuma -- a medicao roda em processador, mais devagar'}")
print("\nAmbiente pronto." if ok else "\nReinicie a sessao e execute esta secao de novo.")

## 3. Repositório e Drive

Clona a `main`, onde estão os 61 pares desde 14/09/2026. Se a medição já constar de
`explicito_bruto.json`, a seção 5 apenas reanalisa. O Drive é montado agora, e não
só na exportação: se a sessão cair depois da medição, a cópia no Drive é o que resta.

In [ ]:
import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

# A branch de trabalho foi integrada e apagada em 14/09/2026. Clonar branch
# inexistente falha, e o notebook ficaria inutilizavel sem aviso.
BRANCH = "main"
REPO = Path("/content/vies-nordeste-bertimbau")

drive.mount("/content/drive")

# Apagar antes de clonar torna a celula repetivel: clonar sobre pasta existente
# falha e deixaria a versao antiga no lugar, sem aviso.
shutil.rmtree(REPO, ignore_errors=True)
r = subprocess.run(["git", "clone", "-q", "--branch", BRANCH,
                    "https://github.com/Aryazinha/vies-nordeste-bertimbau.git", str(REPO)],
                   capture_output=True, text=True)
assert r.returncode == 0, f"clone falhou -- a branch {BRANCH} existe no GitHub?\n{r.stderr}"
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"branch {BRANCH}, commit {COMMIT}")

EXP = REPO / "experimentos"
os.chdir(EXP)
if str(EXP) not in sys.path:
    sys.path.insert(0, str(EXP))
# O Python memoriza que um caminho nao existia; sem limpar o cache, um import
# tentado antes do clone continuaria falhando mesmo com a pasta no lugar.
importlib.invalidate_caches()
import teste_explicito as te
te = importlib.reload(te)

assert len(te.CALIBRACAO_V2) == 61, f"esperados 61 pares, ha {len(te.CALIBRACAO_V2)}"
BRUTO = EXP / "resultados" / "dados" / "explicito_bruto.json"
bruto_antes = json.loads(BRUTO.read_text(encoding="utf-8"))
condicoes_antes = {r["condicao"] for r in bruto_antes}
print(f"{len(bruto_antes)} medicoes ja guardadas, em {len(condicoes_antes)} condicoes")
if "calibracao_v2" in condicoes_antes:
    print("AVISO: calibracao_v2 ja consta como medida; a secao 5 apenas reanalisara")

## 4. Conferência de reprodutibilidade

Remede os cinco pares de `controle_neutro` — 140 medições — e compara com os valores
guardados. **Se o resultado for `NAO REPRODUZ`**, ponha `USAR_CPU = True` no início da
célula e execute esta seção de novo; a medição passa a usar o processador, como a
original.

In [ ]:
from teste_construcional import medir
from teste_sensibilidade import ATRIBUTOS, CONDICOES

USAR_CPU = False
if USAR_CPU:
    # O Medidor escolhe o dispositivo consultando esta funcao ao ser criado.
    torch.cuda.is_available = lambda: False

TOLERANCIA = 1e-3   # em log-probabilidade por token

t0 = time.time()
remedida = medir({"controle_neutro": CONDICOES["controle_neutro"]})
guardada = {(r["condicao"], r["par"], r["moldura"], r["atributo"]): r
            for r in bruto_antes if r["condicao"] == "controle_neutro"}

difs = []
for r in remedida:
    g = guardada.get((r["condicao"], r["par"], r["moldura"], r["atributo"]))
    if g is not None:
        difs.append(max(abs(r["pll_a"] - g["pll_a"]), abs(r["pll_b"] - g["pll_b"])))

assert difs, "nenhuma medicao comparavel -- as molduras ou os atributos mudaram?"
if len(difs) < len(remedida):
    print(f"AVISO: so {len(difs)} de {len(remedida)} medicoes tinham correspondente guardado")
DIF_MAX = max(difs)
REPRODUZ = DIF_MAX <= TOLERANCIA
DISPOSITIVO = "cpu" if (USAR_CPU or GPU_NOME is None) else GPU_NOME

print(f"\n{len(difs)} medicoes comparadas em {time.time() - t0:.0f} s, em {DISPOSITIVO}")
print(f"maior diferenca: {DIF_MAX:.2e} (tolerancia {TOLERANCIA:.0e})")
print("\nREPRODUZ -- siga para a secao 5" if REPRODUZ else
      "\nNAO REPRODUZ -- ponha USAR_CPU = True no inicio desta celula e execute-a de novo")

## 5. Medição dos 61 pares e reanálise do passo 5.4

Executa `teste_explicito.py` inteiro: mede a condição ausente, grava as medições junto
das antigas e regera `explicito_tabelas.md` e `explicito_pares.json`, já sem a
duplicata. Ao final confere que as medições antigas ficaram intactas e que nenhum par
ficou sem medição.

In [ ]:
assert REPRODUZ, "a secao 4 nao reproduziu as medicoes guardadas; nao misture ambientes"

t0 = time.time()
te.main()
print(f"\nmedicao e reanalise em {time.time() - t0:.0f} s")

bruto_depois = json.loads(BRUTO.read_text(encoding="utf-8"))
assert bruto_depois[:len(bruto_antes)] == bruto_antes, "medicoes antigas foram alteradas"

novas = [r for r in bruto_depois if r["condicao"] == "calibracao_v2"]
por_par = {}
for r in novas:
    por_par[r["par"]] = por_par.get(r["par"], 0) + 1
esperadas = sum(len(v) for v in ATRIBUTOS.values())
assert set(por_par) == set(range(61)), f"pares sem medicao: {sorted(set(range(61)) - set(por_par))}"
print(f"{len(novas)} medicoes novas em {len(por_par)} pares "
      f"(de {min(por_par.values())} a {max(por_par.values())} por par; esperadas {esperadas})")

## 6. Reanálise do passo 5.5 — direção do efeito

Lê as mesmas medições, sem nova passagem pelo modelo. O desvio-padrão do grupo de
referência no eixo de caráter, impresso na primeira tabela, é o insumo de
`meta_pares_minimos.py` e deve ser anotado.

In [ ]:
import analise_valencia
analise_valencia = importlib.reload(analise_valencia)
analise_valencia.main()

## 7. Registro do ambiente

Grava em `ambiente_calibracao_v2.json` com que versões, em que dispositivo e com que
resultado de conferência as medições novas foram feitas. Sem isso, o arquivo de
medições misturaria dois ambientes sem registro de qual é qual.

In [ ]:
AMBIENTE = EXP / "resultados" / "dados" / "ambiente_calibracao_v2.json"
AMBIENTE.write_text(json.dumps({
    "data": time.strftime("%Y-%m-%d"),
    "branch": BRANCH,
    "commit": COMMIT,
    "condicao_medida": "calibracao_v2",
    "pares": len(por_par),
    "medicoes": len(novas),
    "dispositivo": DISPOSITIVO,
    "versoes": VERSOES,
    "conferencia_reprodutibilidade": {
        "condicao": "controle_neutro",
        "medicoes_comparadas": len(difs),
        "maior_diferenca": DIF_MAX,
        "tolerancia": TOLERANCIA,
        "ambiente_original": "torch 2.13.0+cpu (requirements-lock.txt)",
    },
}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(AMBIENTE.read_text(encoding="utf-8"))

## 8. Exportação

Compacta os arquivos alterados **com o caminho relativo à raiz do repositório**, grava
uma cópia no Drive, em `MyDrive/medicao_calibracao_v2/`, e oferece o download.

In [ ]:
import zipfile

ARQUIVOS = [
    "experimentos/resultados/dados/explicito_bruto.json",
    "experimentos/resultados/dados/explicito_pares.json",
    "experimentos/resultados/dados/ambiente_calibracao_v2.json",
    "experimentos/resultados/tabelas/explicito_tabelas.md",
    "experimentos/resultados/tabelas/valencia_tabelas.md",
]
ZIP = Path("/content/medicao_calibracao_v2.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for arquivo in ARQUIVOS:
        z.write(REPO / arquivo, arquivo)

DESTINO = Path("/content/drive/MyDrive/medicao_calibracao_v2")
DESTINO.mkdir(parents=True, exist_ok=True)
shutil.copy(ZIP, DESTINO / ZIP.name)
print(f"{len(ARQUIVOS)} arquivos em {ZIP.name}; copia no Drive em {DESTINO}")

from google.colab import files
files.download(str(ZIP))

## Ao terminar

1. Descompacte `medicao_calibracao_v2.zip` **na raiz do repositório local**,
   substituindo os arquivos existentes. Os caminhos internos já apontam para os
   lugares certos.
2. Avise na sessão de trabalho. O que resta é local: regerar `pares_minimos.json` com
   `empacotar_pares.py` e conferir com `--verificar`, atualizar `DP_RUIDO` e
   `REFERENCIA_ATUAL` em `meta_pares_minimos.py`, e corrigir os números citados nos
   documentos (`docs/pendencias.md` 2.8).